In [0]:
# Kafka connection config
# Replace with your Confluent Cloud credentials
KAFKA_BOOTSTRAP_SERVERS = "YOUR_BOOTSTRAP_SERVER"
KAFKA_API_KEY = "YOUR_API_KEY"
KAFKA_API_SECRET = "YOUR_API_SECRET"
KAFKA_TOPIC = "nyc-taxi-trips"

print("Kafka config loaded")

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# define schema for NYC taxi trip JSON
schema = StructType([
    StructField("vendor_id", StringType()),
    StructField("pickup_datetime", StringType()),
    StructField("dropoff_datetime", StringType()),
    StructField("passenger_count", StringType()),
    StructField("trip_distance", StringType()),
    StructField("pickup_longitude", StringType()),
    StructField("pickup_latitude", StringType()),
    StructField("dropoff_longitude", StringType()),
    StructField("dropoff_latitude", StringType()),
    StructField("fare_amount", StringType()),
    StructField("tip_amount", StringType()),
    StructField("total_amount", StringType()),
    StructField("payment_type", StringType()),
    StructField("ingested_at", StringType()),
    StructField("batch_num", StringType())
])

# read from Kafka as a streaming DataFrame
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";')
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

# parse the JSON value from Kafka message
parsed_stream = (
    raw_stream
    .select(
        col("offset"),
        col("partition"),
        col("timestamp").alias("kafka_timestamp"),
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select("offset", "partition", "kafka_timestamp", "data.*")
    .withColumn("bronze_ingested_at", current_timestamp())
)

print("Streaming reader defined")

In [0]:
# write stream to bronze Delta table with checkpointing
bronze_query = (
    parsed_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/nyc_taxi/bronze/checkpoints/nyc_taxi_bronze")
    .trigger(availableNow=True)
    .toTable("nyc_taxi.bronze.nyc_taxi_trips")
)

bronze_query.awaitTermination()
print("Bronze streaming batch complete")
print(f"Query ID: {bronze_query.id}")

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS nyc_taxi")
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi.bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS nyc_taxi.bronze.checkpoints")

In [0]:
spark.sql("SELECT count(*) FROM nyc_taxi.bronze.nyc_taxi_trips").display()

In [0]:
spark.sql("SELECT * FROM nyc_taxi.bronze.nyc_taxi_trips LIMIT 5").display()